In [ ]:
# realtext_grid.ipynb -- run the real-text grid eval on this pod.
#
# Per FULL-N combo it measures the ZERO-SHOT article->game name recall and the
# score after a contrastive linear fine-tune (5-fold CV, every game tested
# once), plus zero-shot tag F1 / drop. One worker process per visible GPU
# (manual shards -- multi-VM is deliberately NOT supported); feature caches
# prefetch into RAM so the distributed FS never stalls the GPU.
#
# Paste the RunPod API key below ONLY on the pod (never commit it) if you want
# the pod to stop itself when the whole grid is done.
RUNPOD_API_KEY_OVERRIDE = ""   # <-- paste on the pod for SELFSTOP
SELFSTOP = True                # stop this pod when EVERY combo has output
SKIP_CONTRASTIVE = False       # True = zero-shot only (much faster)
PREFETCH_WORKERS = 8           # parallel npz readers into RAM (0 = off)
OVERWRITE = False              # True = recompute combos that already have output

print('selfstop:', SELFSTOP, '| contrastive:', not SKIP_CONTRASTIVE)


In [ ]:
# Launch one shard per GPU and monitor until they exit.
import json, os, subprocess, sys, time
from pathlib import Path

REPO = Path('/workspace/stable-query-latent')
subprocess.run(['git', '-C', str(REPO), 'pull'], check=False)

r = subprocess.run(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
                   capture_output=True, text=True)
gpus = [l.strip() for l in r.stdout.splitlines() if l.strip()]
assert gpus, 'no GPUs visible'
logdir = Path('/workspace/stable_query_latent_logs')
logdir.mkdir(parents=True, exist_ok=True)

# Warmup FIRST and synchronously: builds the shared article-embedding cache
# once, so the N shards below all load it instead of racing to embed it.
print('warmup: building the shared article cache (once) ...', flush=True)
w = subprocess.run([sys.executable, '-u', 'VICReg_review/realtext_grid_eval.py',
                    '--warmup-only'], cwd=str(REPO),
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print((w.stdout or '')[-2000:])
assert w.returncode == 0, 'warmup failed -- see output above'

procs, logs = [], []
for i, g in enumerate(gpus):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=g)
    if RUNPOD_API_KEY_OVERRIDE:
        env['RUNPOD_API_KEY'] = RUNPOD_API_KEY_OVERRIDE
    cmd = [sys.executable, '-u', 'VICReg_review/realtext_grid_eval.py',
           '--shard', f'{i}/{len(gpus)}',
           '--prefetch-workers', str(PREFETCH_WORKERS)]
    if SELFSTOP:
        cmd.append('--selfstop')
    if SKIP_CONTRASTIVE:
        cmd.append('--skip-contrastive')
    if OVERWRITE:
        cmd.append('--overwrite')
    lp = logdir / f'realtext_{i}.log'
    lf = open(lp, 'a')
    procs.append(subprocess.Popen(cmd, cwd=str(REPO), env=env,
                                  stdout=lf, stderr=subprocess.STDOUT))
    logs.append(lp)
    print(f'shard {i}/{len(gpus)} on gpu {g} -> {lp}')

summary = REPO / 'VICReg_review/heads/cloud_full_sweep_a100/realtext_grid_metrics.json'
while any(p.poll() is None for p in procs):
    time.sleep(60)
    try:
        rows = json.loads(summary.read_text(encoding='utf-8'))['n_rows']
    except Exception:
        rows = 0
    alive = sum(p.poll() is None for p in procs)
    print(f'{time.strftime("%H:%M:%S")}  shards alive={alive}  summary rows={rows}',
          flush=True)

print('all shards exited:', [p.returncode for p in procs])
for p, lp in zip(procs, logs):
    if p.returncode != 0:
        tail = lp.read_text(encoding='utf-8', errors='replace').splitlines()[-15:]
        print(f'!! shard log tail ({lp}):')
        for line in tail:
            print('   ', line)


In [ ]:
# Champion re-selection by fine-tuned name recall. Runs only if the shards
# finished cleanly AND the pod did not stop itself (SELFSTOP cuts the session
# before this cell); in that case run the same command locally after
# sync_results pulls realtext_grid_metrics.json.
import subprocess, sys
from pathlib import Path

REPO = Path('/workspace/stable-query-latent')
if all(p.returncode == 0 for p in procs):
    subprocess.run([sys.executable, 'VICReg_review/get_champions_namerank.py',
                    '--by', 'namerank_con'], cwd=str(REPO))
else:
    print('some shards failed -- fix and re-run (resume skips finished combos).')
